# Lifetime PD — the evidence

Every statistic below is a tested function in `creditsurv`. This notebook calls them
and shows what came back; it contains no statistics of its own. Read
[`docs/variable_selection.md`](../docs/variable_selection.md) for the procedure and
the thresholds, and [`docs/data_dictionary.md`](../docs/data_dictionary.md) for what
the fields mean.

The **fit itself is not here**. On the whole population it is tens of minutes, so it
is a CLI step — `creditsurv fit` and `creditsurv report` — and this notebook reads
what they leave behind. That is the same division as
[`01_portfolio.ipynb`](01_portfolio.ipynb): the package computes, the commands
persist, the notebook shows.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Image

from creditsurv.config import MACRO_CANDIDATES, ORDINAL, STATIC_CONTINUOUS, reports_dir
from creditsurv.data.fred import load_macro_panel
from creditsurv.data.panel import WEIGHT, cells_to_episodes
from creditsurv.data.store import load_cells
from creditsurv.explore import (
    collinear_pairs,
    curves_cross,
    default_rate_by_band,
    survival_by_stratum,
    weighted_correlation,
)
from creditsurv.features import BIN_EDGES, bin_covariates
from creditsurv.models.selection import variance_inflation

warnings.simplefilter("ignore", FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

episodes = cells_to_episodes(load_cells(), load_macro_panel())
print(f"{len(episodes):,} cells")
print(f"{int(episodes[WEIGHT].sum()):,} loan-months")
print(f"{int(episodes.loc[episodes['event'], WEIGHT].sum()):,} defaults")
print(f"ages {episodes['age'].min()} to {episodes['age'].max()} months")

## 1. Can one survival function serve?

The project's commitment is to **a single survival function**, with heterogeneity
carried by covariates rather than by segmenting the book. That is an assumption, and
this is the check that can refuse it before any model is fitted.

Curves that separate and stay separated are exactly what a model with covariates is
for: the covariate shifts the scale and one functional form serves. Curves that
**cross** cannot be reconciled by scaling one into the other, and no coefficient will
fix it.

Kaplan-Meier over individual loans is not available on an aggregated panel. The curve
below is the same construction — an empirical hazard per age, chained — computed on
exposure, which is what the estimator reduces to on weighted cells.

In [ ]:
for stratum in ("purpose", "occupancy", "term_years"):
    curves = survival_by_stratum(episodes, stratum)
    crossing = curves_cross(curves)
    final = (
        curves.sort_values("age")
        .groupby("stratum", observed=True)
        .last()[["survival", "exposure"]]
    )
    print(f"{stratum}: curves cross = {crossing}")
    print(final.round(4).to_string())
    print()

## 2. Does each covariate order the risk?

The rate is **events over exposure** — a monthly hazard — not defaults over loans.
Two bands can hold the same number of defaults and differ entirely in risk if one was
watched ten times as long, and across twenty-seven vintages that is the rule rather
than the exception.

What to look for is monotonicity. A covariate whose default rate rises and falls
across its own bands is either mis-binned or proxying something else — which is how
the untreated `999` sentinel in estimated LTV was found.

In [ ]:
# The origination covariates are already carried at their band's midpoint by the
# aggregation. The macro-derived ones are rebuilt continuous, so they are banded here
# -- from the same BIN_EDGES the model's binned formula would use.
banded = bin_covariates(
    episodes, edges={name: BIN_EDGES[name] for name in ("cltv_drift", "rate_gap", "hpi_growth")}
)

for name, frame in (
    ("fico_s", episodes),
    ("orig_ltv", episodes),
    ("dti", episodes),
    ("term_years", episodes),
    ("cltv_drift_binned", banded),
    ("rate_gap_binned", banded),
    ("hpi_growth_binned", banded),
):
    table = default_rate_by_band(frame, name)
    rate = table["default_rate"]
    monotone = rate.is_monotonic_increasing or rate.is_monotonic_decreasing
    print(f"{name}  (monotone: {monotone})")
    print(table.round(6).to_string(index=False))
    print()

## 3. Collinearity

Seventeen candidates, of which thirteen come from fourteen macroeconomic series. Five
of those series are interest rates, and as *time series* they move as one thing over
1999–2026, so this is where the specification was expected to fail.

It does not, and the reason is the panel structure rather than luck: the design matrix
is indexed by vintage and age rather than calendar time, and the gap-since-origination
covariates project onto that second dimension. `docs/variable_selection.md` sets out
the argument and the corollary — replace the gaps with levels and the collinearity
returns at full strength.

In [ ]:
candidates = [*STATIC_CONTINUOUS, *ORDINAL, *MACRO_CANDIDATES]

correlation = weighted_correlation(episodes, candidates, weight=WEIGHT)
print(f"pairs above |rho| = 0.8: {len(collinear_pairs(correlation, threshold=0.8))}")

pairs = (
    correlation.stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "first", "level_1": "second"})
)
pairs = pairs[pairs["first"] < pairs["second"]]
pairs = pairs.reindex(pairs["correlation"].abs().sort_values(ascending=False).index)

print()
print("strongest pairs:")
print(pairs.head(8).round(3).to_string(index=False))

In [ ]:
vif = variance_inflation(episodes, candidates, weight=WEIGHT)
print(f"covariates above VIF = 10: {(vif['vif'] > 10).sum()}")
vif.round(2)

## 4. The fitted model

From `creditsurv fit`, which persists its coefficient table because a fit on this
table is tens of minutes and nothing downstream should pay for one twice.

This is an **accelerated failure time** model, so a coefficient acts on the logarithm
of survival *time*: positive lengthens expected survival and therefore **lowers**
risk. `exp(coef)` is a **time ratio, not a hazard ratio** — reading it as one inverts
the sign of every conclusion.

In [ ]:
coefficients = Path(reports_dir()) / "coefficients.csv"

if coefficients.exists():
    table = pd.read_csv(coefficients, index_col=[0, 1])
    display(table.round(4))
else:
    print(f"No coefficients at {coefficients}. Run: uv run creditsurv fit")

## 5. Against the non-parametric estimate

The strongest evidence available, because the Kaplan-Meier curve assumes nothing
about the distribution. A fitted curve straying outside its confidence band is being
contradicted by the data rather than merely smoothing it.

The curve is chained from the model's monthly hazard along each cell's *realised*
covariate path, not predicted from origination covariates and averaged. The
difference is not cosmetic: `cltv_drift` and `unemp_gap` are zero at origination by
construction, so freezing them there assumes house prices never move and unemployment
never changes. An earlier version of this comparison did exactly that and put only
half the horizon inside the band.

From `creditsurv report`.

In [ ]:
figure = Path(reports_dir()) / "figures" / "survival_vs_km.png"

if figure.exists():
    display(Image(str(figure)))
else:
    print(f"No figure at {figure}. Run: uv run creditsurv report")